# CATI Phase 2 — End-to-End Fine-tuning

Jointly trains YOLOv11s backbone + CATI FiLM conditioning using detection loss.

| Step | Cell | Notes |
|------|------|-------|
| 0. Setup | 1 | Mount Drive, clone repo, install deps |
| 1. Prerequisites | 2 | Check Phase 1 ckpt + YOLO dataset |
| 2. Layer verification | 3 | Confirm neck FiLM layer indices |
| 3. Phase 2 training | 4 | Backbone + neck FiLM + aux loss |
| 4. mAP comparison | 5 | CATI vs baseline |
| 5. Ablation | 6 | YOLO fine-tune without CATI |
| 6. Confidence sweep | 7 | Find optimal confidence threshold |

**Improvements over v1:** neck FiLM (PAN conditioning), auxiliary context regularization loss.

In [ ]:
# Cell 1: Mount Drive + setup
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_DIR = '/content/sg-smart-city-analytics'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Suhxs-Reddy/sg-smart-city-analytics.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
    for k in list(sys.modules.keys()):
        if k.startswith('src.'): del sys.modules[k]

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

!pip install -q ultralytics torch torchvision pyyaml pillow

import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory/2**30:.1f} GB VRAM)')

FEATURE_DIR  = '/content/drive/MyDrive/sg_smart_city/data/features'
YOLO_DIR     = '/content/drive/MyDrive/sg_smart_city/data/yolo_dataset'
MODEL_DIR    = '/content/drive/MyDrive/sg_smart_city/models'
PHASE1_CKPT  = f'{MODEL_DIR}/cati_best.pt'
PHASE2_DIR   = f'{MODEL_DIR}/phase2'

In [ ]:
# Cell 2: Verify prerequisites
from pathlib import Path

checks = {
    'Phase 1 checkpoint': Path(PHASE1_CKPT).exists(),
    'YOLO dataset':        Path(YOLO_DIR).exists(),
    'data.yaml':           (Path(YOLO_DIR)/'data.yaml').exists(),
    'Train images':        len(list((Path(YOLO_DIR)/'images'/'train').glob('*'))) > 0,
    'Train labels':        len(list((Path(YOLO_DIR)/'labels'/'train').glob('*.txt'))) > 0,
}
for name, ok in checks.items():
    print(f'  {"ok" if ok else "MISSING"} {name}')

if not all(checks.values()):
    raise RuntimeError('Prerequisites not met — check above')
print('\nAll prerequisites met.')

In [ ]:
# Cell 3: Verify neck layer indices
# Prints shapes for YOLO layers 10-25.
# Look for 3 layers marked as neck FiLM candidates with shapes:
#   (1, 128, 80, 80)  <- neck P3
#   (1, 256, 40, 40)  <- neck P4
#   (1, 512, 20, 20)  <- neck P5
# Update NECK_HOOK_LAYERS below if they differ from [16, 19, 22].
from src.training.train_phase2 import CATIPhase2Trainer

shapes = CATIPhase2Trainer.verify_layers('yolo11s.pt')

NECK_HOOK_LAYERS = [16, 19, 22]  # update after checking output above
print(f'\nUsing neck hook layers: {NECK_HOOK_LAYERS}')

In [ ]:
# Cell 4: Phase 2 Training
import logging, os
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', force=True)
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_SILENT'] = 'true'

from src.training.train_phase2 import CATIPhase2Trainer

USE_NECK_FILM = True  # set False to train backbone FiLM only

trainer = CATIPhase2Trainer(
    yolo_dataset_dir=YOLO_DIR,
    feature_dir=FEATURE_DIR,
    cati_weights_path=PHASE1_CKPT,
    model_variant='yolo11s',
    epochs=25,
    batch_size=8,
    lr=1e-4,
    device='cuda',
    freeze_backbone_epochs=3,
    save_dir=PHASE2_DIR,
    use_neck_film=USE_NECK_FILM,
    neck_hook_layers=NECK_HOOK_LAYERS,
)

results = trainer.train()
print('Phase 2 complete!')
print(f'Checkpoints saved to: {PHASE2_DIR}')

In [ ]:
# Cell 5: Evaluate — CATI vs baseline mAP
from ultralytics import YOLO
from pathlib import Path

data_yaml = str(Path(YOLO_DIR) / 'data.yaml')

print('Evaluating baseline YOLOv11s (pretrained, no fine-tune)...')
baseline = YOLO('yolo11s.pt')
baseline_metrics = baseline.val(data=data_yaml, imgsz=640, device='cuda', verbose=False)
print(f'Baseline mAP50:    {baseline_metrics.box.map50:.4f}')
print(f'Baseline mAP50-95: {baseline_metrics.box.map:.4f}')

phase2_best = next(Path(PHASE2_DIR).glob('**/best.pt'), None)
if phase2_best:
    print(f'\nEvaluating CATI Phase 2 ({phase2_best.name})...')
    cati_model = YOLO(str(phase2_best))
    cati_metrics = cati_model.val(data=data_yaml, imgsz=640, device='cuda', verbose=False)
    print(f'CATI mAP50:        {cati_metrics.box.map50:.4f}')
    print(f'CATI mAP50-95:     {cati_metrics.box.map:.4f}')
    delta = cati_metrics.box.map50 - baseline_metrics.box.map50
    print(f'DeltamAP50 (CATI - baseline): {delta:+.4f}')
else:
    print('No Phase 2 best.pt found — run Cell 4 first')

In [ ]:
# Cell 6: Ablation — fine-tune plain YOLOv11s WITHOUT CATI
# Isolates how much mAP gain is domain adaptation vs CATI conditioning.
# Compare this mAP against Cell 5 CATI result to get the true CATI delta.
from ultralytics import YOLO
from pathlib import Path
import os
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_SILENT'] = 'true'

ABLATION_DIR = f'{MODEL_DIR}/ablation_no_cati'
data_yaml = str(Path(YOLO_DIR) / 'data.yaml')

print('Fine-tuning plain YOLOv11s (no CATI)...')
ablation_model = YOLO('yolo11s.pt')
ablation_model.train(
    data=data_yaml,
    epochs=20,
    batch=8,
    imgsz=640,
    lr0=1e-4,
    lrf=0.01,
    warmup_epochs=3,
    device='cuda',
    project=ABLATION_DIR,
    name='yolo_no_cati',
    save=True,
    verbose=False,
    workers=0,
)

ablation_best = next(Path(ABLATION_DIR).glob('**/best.pt'), None)
if ablation_best:
    ablation_val = YOLO(str(ablation_best)).val(
        data=data_yaml, imgsz=640, device='cuda', verbose=False
    )
    print(f'Ablation mAP50:    {ablation_val.box.map50:.4f}')
    print(f'Ablation mAP50-95: {ablation_val.box.map:.4f}')
    try:
        true_cati_delta = cati_metrics.box.map50 - ablation_val.box.map50
        print(f'True CATI delta (CATI - ablation): {true_cati_delta:+.4f}')
        print('(positive = FiLM conditioning adds real value)')
    except NameError:
        print('Run Cell 5 first to compute cati_metrics')

In [ ]:
# Cell 7: Confidence threshold sweep
from ultralytics import YOLO
from pathlib import Path

phase2_best = next(Path(PHASE2_DIR).glob('**/best.pt'), None)
if not phase2_best:
    raise RuntimeError('Run Cell 4 first')

data_yaml = str(Path(YOLO_DIR) / 'data.yaml')
model = YOLO(str(phase2_best))

thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
print(f'Sweeping confidence thresholds on {phase2_best.name}...')
print(f'{"Conf":<8} {"P":<8} {"R":<8} {"mAP50":<10} {"F1"}')
print('-' * 46)

best_rows = []
for conf in thresholds:
    m = model.val(data=data_yaml, imgsz=640, conf=conf, device='cuda', verbose=False)
    p = m.box.mp
    r = m.box.mr
    map50 = m.box.map50
    f1 = 2 * p * r / (p + r + 1e-8)
    best_rows.append((conf, p, r, map50, f1))
    print(f'{conf:<8.2f} {p:<8.3f} {r:<8.3f} {map50:<10.4f} {f1:.3f}')

best_map = max(best_rows, key=lambda x: x[3])
best_f1  = max(best_rows, key=lambda x: x[4])
print(f'\nBest mAP50 at conf={best_map[0]:.2f}: {best_map[3]:.4f}')
print(f'Best F1    at conf={best_f1[0]:.2f}:  {best_f1[4]:.3f}')